# # Xplore: A LangSmith Example for PDF and YouTube Content

This notebook demonstrates how to use LangSmith to load and process content from both PDF documents and YouTube videos. It includes:
- Loading PDF files with OCR fallback for image-based content
- Downloading YouTube video converted to audio and transcribing it using Faster Whisper
- Combining and chunking the extracted content for further processing

# # PDF Loading with OCR Fallback

In [3]:
from langchain_core.messages.block_translators import groq
from sqlalchemy.orm import persistence
!pip install langchain --quiet
!pip install langchain-core --quiet
!pip install langchain-community --quiet
!pip install pypdf --quiet
!pip install pytesseract --quiet
!pip install pillow --quiet
!pip install pdf2image --quiet
!brew install tesseract --quiet 2>/dev/null || echo "Tesseract may need manual install"
!brew install poppler --quiet 2>/dev/null || echo "Poppler may need manual install"

⠋ JSON API formula.jws.json                          Downloading  33.2MB/-------
⠋ JSON API cask.jws.json                             Downloading  16.9MB/-------⠋ JSON API formula.jws.json                          Downloading  33.2MB/-------
⠋ JSON API cask.jws.json                             Downloading  16.9MB/-------⠙ JSON API formula.jws.json                          Downloading   4.3MB/-------
⠙ JSON API cask.jws.json                             Downloading  16.9MB/-------⠚ JSON API formula.jws.json                          Downloading  12.1MB/-------
⠚ JSON API cask.jws.json                             Downloaded   16.9MB/-------✔︎ JSON API cask.jws.json                             Downloaded   16.9MB/ 16.9MB
⠚ JSON API formula.jws.json                          Downloading  16.1MB/-------⠞ JSON API formula.jws.json                          Downloading  19.4MB/-------⠞ JSON API formula.jws.json                          Downloading  23.3MB/-------⠖ JSON API formula.jws.json       

In [4]:
import pytesseract
import os
from PIL import Image
from pdf2image import convert_from_path
from langchain_community.document_loaders import PyPDFLoader, YoutubeLoader
from langchain_core.documents import Document


def load_pdf_with_ocr(pdf_path):
    """
    Load PDF and extract text from both text-based and image-based pages using OCR.
    Falls back to OCR if standard text extraction returns empty content.
    """
    documents = []

    # First try standard PDF loader
    try:
        loader = PyPDFLoader(pdf_path)
        documents = loader.load()
    except Exception as e:
        print(f"Standard PDF loading failed: {e}")
        documents = []

    # Check if pages have meaningful content, if not use OCR
    has_content = any(len(doc.page_content.strip()) > 100 for doc in documents)

    if not has_content:
        print("No text found in standard extraction, trying OCR...")
        try:
            # Convert PDF pages to images and extract text using OCR
            images = convert_from_path(pdf_path)
            ocr_documents = []

            for page_num, image in enumerate(images):
                # Extract text using Tesseract OCR
                text = pytesseract.image_to_string(image)

                # Create a document for this page
                doc = Document(
                    page_content=text,
                    metadata={
                        "source": pdf_path,
                        "page": page_num,
                        "extraction_method": "OCR"
                    }
                )
                ocr_documents.append(doc)

            documents = ocr_documents
            print(f"Successfully extracted {len(documents)} pages using OCR")
        except Exception as e:
            error_msg = str(e)
            print(f"OCR extraction failed: {e}")
            if "poppler" in error_msg.lower() or "page count" in error_msg.lower():
                print("\n❌ Poppler is not installed or not in PATH!")
                print("Fix this by running: brew install poppler")
                print("\nIf Tesseract is also missing, run: brew install tesseract")
            elif "tesseract" in error_msg.lower():
                print("\n❌ Tesseract is not installed!")
                print("Fix this by running: brew install tesseract")
            else:
                print("Please ensure both Tesseract and Poppler are installed on your system.")

    return documents


In [5]:
# Usage
pdf_path = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf"
# pdf_path = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/audhil-report.pdf"
pdf_pages = load_pdf_with_ocr(pdf_path)
if pdf_pages:
    print(f"Loaded {len(pdf_pages)} pages")
    print("First page content preview:")
    print(pdf_pages[0].page_content[:1500])
    #print(pdf_pages[0])

Loaded 209 pages
First page content preview:
Embedded Systems Design: An Introduction to Processes, Tools, and 
Techniques 
by Arnold S. Berger ISBN: 1578200733 
CMP Books © 2002 (237 pages) 
An easy-to-understand guidebook for those embarking upon an embedded 
processor development project.  
 
 
Table of Contents  
 
 
Embedded Systems Design—An Introduc tion to Processes, Tools, and 
Techniques  
 Preface  
 Introduction  
 Chapter 1 - The Embedded Design Life Cycle 
 Chapter 2 - The Selection Process 
 Chapter 3 - The Partitioning Decision 
 Chapter 4 - The Development Environment 
 Chapter 5 - Special Software Techniques 
 Chapter 6 - A Basic Toolset 
 Chapter 7 - BDM, JTAG, and Nexus 
 Chapter 8 - The ICE — An Integrated Solution 
 Chapter 9 - Testing 
 Chapter 10 - The Future 
 Index  
 List of Figures  
 List of Tables  
 List of Listings  
 List of Sidebars  
 
TEAMFLY
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
Tea

# # Youtube video -> audio -> transcript

In [6]:
# Fix PATH for Homebrew tools (especially important for notebook environments)
import os
import subprocess

# Ensure homebrew bin is in PATH
homebrew_paths = ["/opt/homebrew/bin", "/usr/local/bin"]
current_path = os.environ.get("PATH", "").split(":")
for path in homebrew_paths:
    if path not in current_path:
        current_path.insert(0, path)
os.environ["PATH"] = ":".join(current_path)

# Install system dependencies for YouTube audio processing
!brew install ffmpeg --quiet 2>/dev/null || echo "FFmpeg may need manual install"
# Install Python dependencies
!pip install yt_dlp --quiet
!pip install --upgrade pip
!pip install pydub --quiet
!pip install faster-whisper --quiet
!pip install torch --quiet
!pip install ffmpeg-python --quiet

In [7]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers.audio import FasterWhisperParser  # to generate transcript from audio
from langchain_community.document_loaders.blob_loaders.youtube_audio import YoutubeAudioLoader

# Set FFmpeg path explicitly for Homebrew installation
os.environ["PATH"] = "/opt/homebrew/bin:" + os.environ.get("PATH", "")

# Also set explicit ffmpeg location for yt-dlp
import subprocess

ffmpeg_path = None
ffprobe_path = None

try:
    ffmpeg_path = subprocess.check_output(["which", "ffmpeg"], text=True).strip()
    ffprobe_path = subprocess.check_output(["which", "ffprobe"], text=True).strip()

    # Set environment variables for the entire process
    os.environ["FFMPEG_LOCATION"] = ffmpeg_path
    os.environ["FFPROBE_LOCATION"] = ffprobe_path
    os.environ["PATH"] = f"{os.path.dirname(ffmpeg_path)}:" + os.environ.get("PATH", "")

    print(f"✓ FFmpeg found at: {ffmpeg_path}")
    print(f"✓ FFprobe found at: {ffprobe_path}")

    # Create yt-dlp config to locate ffmpeg
    yt_dlp_config_dir = os.path.expanduser("~/.config/yt-dlp")
    os.makedirs(yt_dlp_config_dir, exist_ok=True)

    yt_dlp_config = os.path.join(yt_dlp_config_dir, "config.txt")
    with open(yt_dlp_config, "w") as f:
        f.write(f"# Auto-generated config\nffmpeg-location {os.path.dirname(ffmpeg_path)}\n")
    print(f"✓ yt-dlp config created at {yt_dlp_config}")

except Exception as e:
    print(f"Error: Could not locate ffmpeg/ffprobe: {e}")
    print("Please install: brew install ffmpeg")


✓ FFmpeg found at: /opt/homebrew/bin/ffmpeg
✓ FFprobe found at: /opt/homebrew/bin/ffprobe
✓ yt-dlp config created at /Users/00156257shakeermohammedaudhil/.config/yt-dlp/config.txt


In [8]:
url = "https://www.youtube.com/watch?v=uFhDGagZzjs"
save_dir = "/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube"

# Create save directory if it doesn't exist
os.makedirs(save_dir, exist_ok=True)

# Load YouTube audio and transcribe
try:
    print(f"Downloading and transcribing: {url}")
    loader = GenericLoader(
        YoutubeAudioLoader([url], save_dir),
        FasterWhisperParser()
    )
    yt_docs = loader.load()
    print(f"\n✓ Successfully loaded {len(yt_docs)} documents from YouTube")
    if yt_docs:
        print("\nTranscription preview:")
        print(yt_docs[0].page_content[:500])
except Exception as e:
    error_str = str(e)
    print(f"\n❌ Error loading YouTube audio: {e}")
    print(f"FFmpeg location: {ffmpeg_path}")
    print("\nTroubleshooting:")
    print("1. Ensure FFmpeg is installed: brew install ffmpeg")
    print("2. Verify PATH: which ffmpeg, which ffprobe")
    print("3. Test FFmpeg: ffmpeg -version")
    if "ffmpeg" in error_str.lower() or "ffprobe" in error_str.lower():
        print("\nThe issue is FFmpeg-related. Try restarting the notebook.")


[youtube] Extracting URL: https://www.youtube.com/watch?v=uFhDGagZzjs
[youtube] uFhDGagZzjs: Downloading webpage


[youtube] uFhDGagZzjs: Downloading android vr player API JSON
[info] uFhDGagZzjs: Downloading 1 format(s): 140
[download] /Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube/Lecture 01： Introduction to Embedded Systems.m4a has already been downloaded
[download] 100% of   27.27MiB
[ExtractAudio] Not converting audio /Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube/Lecture 01： Introduction to Embedded Systems.m4a; file is already in target format m4a


[2026-06-07 16:45:45.671] [ctranslate2] [thread 2675014] [warning] The compute type inferred from the saved model is float16, but the target device or backend do not support efficient float16 computation. The model weights have been automatically converted to use the float32 compute type instead.



✓ Successfully loaded 219 documents from YouTube

Transcription preview:
 Let me welcome you to this course which will be conducted jointly by myself and Doctor


In [9]:
len(yt_docs)

219

In [10]:
len(pdf_pages)

209

In [11]:
combined_docs = pdf_pages + yt_docs
print(f"Total combined documents: {len(combined_docs)}")

Total combined documents: 428


In [12]:
## Chunking

In [13]:
!pip install langchain_text_splitters --quiet

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

chunk_size = 1024
chunk_overlap = 200
splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
chunked_docs = splitter.split_documents(combined_docs)



In [15]:
chunked_docs[0]

Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'creator': 'PyPDF', 'creationdate': '1999-05-17T00:46:04+00:00', 'moddate': '2004-07-22T18:57:31+03:00', 'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf', 'total_pages': 209, 'page': 0, 'page_label': '1'}, page_content='Embedded Systems Design: An Introduction to Processes, Tools, and \nTechniques \nby Arnold S. Berger ISBN: 1578200733 \nCMP Books © 2002 (237 pages) \nAn easy-to-understand guidebook for those embarking upon an embedded \nprocessor development project.  \n \n \nTable of Contents  \n \n \nEmbedded Systems Design—An Introduc tion to Processes, Tools, and \nTechniques  \n Preface  \n Introduction  \n Chapter 1 - The Embedded Design Life Cycle \n Chapter 2 - The Selection Process \n Chapter 3 - The Partitioning Decision \n Chapter 4 - The Development Environment \n Chapter 5 - Special Software Techniques \n Chapter 6 - A Basic Toolset \n Chapter 7 - BDM, JTAG, and Ne

In [16]:
print(f"Total chunked documents: {len(chunked_docs)}")

Total chunked documents: 897


# # Embeddings

In [17]:
!pip install -U langchain-huggingface
!pip install sentence-transformers
from langchain_community.embeddings import HuggingFaceEmbeddings

all_minilm_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
multilingual_embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-large")

/var/folders/gb/6cys8vkd0_1dzs4_k2h5pwp00000gn/T/ipykernel_46912/224091864.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  all_minilm_embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [18]:
sentence1 = 'I like India, and it is my motherland'
sentence2 = 'I like Malaysia, it lies in east of India'
sentence3 = 'One of the best places to visit in India is the Taj Mahal'

In [19]:
embedding1 = all_minilm_embeddings.embed_query(sentence1)
embedding2 = all_minilm_embeddings.embed_query(sentence2)
embedding3 = all_minilm_embeddings.embed_query(sentence3)

In [20]:
print(f"Embedding 1 length: {len(embedding1)}")
print(f"Embedding 2 length: {len(embedding2)}")
print(f"Embedding 3 length: {len(embedding3)}")

Embedding 1 length: 384
Embedding 2 length: 384
Embedding 3 length: 384


In [21]:
import numpy as np

np.dot(embedding1, embedding2)

np.float64(0.6931444003292835)

In [22]:
np.dot(embedding1, embedding3)

np.float64(0.45283075438113884)

# # Vector DB

In [23]:
# understand HNSW algorithm and how to use it with langchain

In [24]:
!pip install chromadb --quiet
!pip install faiss-cpu --quiet
from langchain_community.vectorstores import Chroma

# persist_directory = '/db/chroma/'
vectordb = Chroma.from_documents(documents=chunked_docs, embedding=multilingual_embeddings)


In [25]:
question = "How to do testing in Embedded Systems?"
# question = "India is my country"

In [26]:
vectordb.similarity_search(question, k=3)  # top 3

[Document(metadata={'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube/Lecture 01： Introduction to Embedded Systems.m4a', 'timestamps': '[1143.28s -> 1156.32s]', 'probability': '96%', 'language': 'en'}, page_content='you respond to the inputs ok. Now how can you define an embedded system based on whatever we'),
 Document(metadata={'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf', 'producer': 'Acrobat Distiller 4.0 for Windows', 'total_pages': 209, 'creator': 'PyPDF', 'moddate': '2004-07-22T18:57:31+03:00', 'page': 156, 'page_label': '157', 'creationdate': '1999-05-17T00:46:04+00:00'}, page_content='Chapter 9: Testing \nEmbedded systems software testing shares much in common with application \nsoftware testing. Thus, much of this chapter is a summary of basic testing \nconcepts and terminology. However, some important differences exist between \napplication testing and embedded systems testing. Embedded developers often \

In [27]:
vectordb.similarity_search_with_score(question, k=3)  # top 3 with scores

[(Document(metadata={'language': 'en', 'timestamps': '[1143.28s -> 1156.32s]', 'probability': '96%', 'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube/Lecture 01： Introduction to Embedded Systems.m4a'}, page_content='you respond to the inputs ok. Now how can you define an embedded system based on whatever we'),
  0.26039421558380127),
 (Document(metadata={'creationdate': '1999-05-17T00:46:04+00:00', 'page': 156, 'page_label': '157', 'moddate': '2004-07-22T18:57:31+03:00', 'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf', 'creator': 'PyPDF', 'total_pages': 209, 'producer': 'Acrobat Distiller 4.0 for Windows'}, page_content='Chapter 9: Testing \nEmbedded systems software testing shares much in common with application \nsoftware testing. Thus, much of this chapter is a summary of basic testing \nconcepts and terminology. However, some important differences exist between \napplication testing and embedded systems testing. E

# # Retrieval

In [28]:
vectordb.max_marginal_relevance_search(question, k=3,
                                       fetch_k=10)  # k = 3 top results, fetch_k = 10 fetch top 10 and then re-rank them to get top 3 with more diversity

[Document(metadata={'timestamps': '[1143.28s -> 1156.32s]', 'probability': '96%', 'language': 'en', 'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/youtube/Lecture 01： Introduction to Embedded Systems.m4a'}, page_content='you respond to the inputs ok. Now how can you define an embedded system based on whatever we'),
 Document(metadata={'producer': 'Acrobat Distiller 4.0 for Windows', 'page_label': '164', 'total_pages': 209, 'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf', 'creator': 'PyPDF', 'creationdate': '1999-05-17T00:46:04+00:00', 'page': 163, 'moddate': '2004-07-22T18:57:31+03:00'}, page_content='chosen based on a guess about what errors are likely. This testing strategy is \nuseful when you’re integrating new functionality with a stable base of legacy code. \nBecause the code base is already well tested, it makes sense to focus your test \nefforts in the area where the new code and the old code come together. \nTestin

# # Meta data filtering

In [29]:
vectordb.similarity_search(question, k=3, filter={
    'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf'})

[Document(metadata={'creator': 'PyPDF', 'source': '/Users/00156257shakeermohammedaudhil/Documents/ZRAG/data/embedded-systems.pdf', 'page_label': '157', 'producer': 'Acrobat Distiller 4.0 for Windows', 'moddate': '2004-07-22T18:57:31+03:00', 'page': 156, 'total_pages': 209, 'creationdate': '1999-05-17T00:46:04+00:00'}, page_content='Chapter 9: Testing \nEmbedded systems software testing shares much in common with application \nsoftware testing. Thus, much of this chapter is a summary of basic testing \nconcepts and terminology. However, some important differences exist between \napplication testing and embedded systems testing. Embedded developers often \nhave access to hardware-based test tools that are generally not used in application \ndevelopment. Also, embedded systems often have unique characteristics that \nshould be reflected in the test plan. These differences tend to give embedded \nsystems testing its own distinctive flavor. This chapter covers the basics of testing \nand te

# # Groq

In [30]:
!pip install langchain-groq --quiet

In [31]:
import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")

In [34]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    max_tokens=250
)

In [36]:
response = llm.invoke("Write about India!")
response.content

"India, a country of vibrant culture, rich history, and breathtaking landscapes, is a land of diverse experiences. Located in South Asia, India is the seventh-largest country in the world, with a population of over 1.3 billion people. It is a federal republic, comprising 28 states and 8 union territories.\n\n**History and Culture**\n\nIndia's history dates back to the Indus Valley Civilization, which flourished around 3300 BCE. The country has been ruled by various empires, including the Mauryan, Gupta, and Mughal dynasties. The Mughal Empire, which ruled from the 16th to the 19th century, left a lasting legacy in the form of stunning architecture, art, and literature.\n\nIndian culture is a unique blend of Hinduism, Buddhism, Jainism, and Islam. The country is home to over 1,600 languages and dialects, with Hindi and English being the official languages. The Indian calendar is based on the lunar cycle, with festivals like Diwali, Holi, and Navratri being celebrated with great fervor.\

# # Prompt Engineering - Role; Instruction; Context; Examples;

In [41]:
system_prompt = ("You are an assistant for question-answering tasks. "
                 "Use the following pieces of retrieved context to answer the question. "
                 "If you don't know the answer, just say that you don't know. "
                 "Use three sentences maximum and keep the answer concise."
                 )

In [42]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

system_message = SystemMessage(content=system_prompt)

In [45]:
docs = vectordb.similarity_search_with_score(question, k=5)

In [48]:
import pandas as pd

_docs = pd.DataFrame(
    [(question, doc[0].page_content, doc[0].metadata.get('source'), doc[0].metadata.get('page'), doc[1]) for doc in
     docs],
    columns=['query', 'paragraph', 'document', 'page_number', 'relevant_score']
)

_docs

,query,paragraph,document,page_number,relevant_score
0,How to do testing in Embedded Systems?,you respond to the inputs ok. Now how can you ...,/Users/00156257shakeermohammedaudhil/Documents...,NaN,0.260394
1,How to do testing in Embedded Systems?,Chapter 9: Testing \nEmbedded systems software...,/Users/00156257shakeermohammedaudhil/Documents...,156.0,0.262700
2,How to do testing in Embedded Systems?,chosen based on a guess about what errors are ...,/Users/00156257shakeermohammedaudhil/Documents...,163.0,0.277319
3,How to do testing in Embedded Systems?,some way to measure how well our tests are cov...,/Users/00156257shakeermohammedaudhil/Documents...,165.0,0.281692
4,How to do testing in Embedded Systems?,Now talking about this embedded systems again ...,/Users/00156257shakeermohammedaudhil/Documents...,NaN,0.283075


In [50]:
context = "\n\n".join(_docs['paragraph'])
context

'you respond to the inputs ok. Now how can you define an embedded system based on whatever we\n\nChapter 9: Testing \nEmbedded systems software testing shares much in common with application \nsoftware testing. Thus, much of this chapter is a summary of basic testing \nconcepts and terminology. However, some important differences exist between \napplication testing and embedded systems testing. Embedded developers often \nhave access to hardware-based test tools that are generally not used in application \ndevelopment. Also, embedded systems often have unique characteristics that \nshould be reflected in the test plan. These differences tend to give embedded \nsystems testing its own distinctive flavor. This chapter covers the basics of testing \nand test case development and points out details unique to embedded systems \nwork along the way. \nWhy Test? \nBefore you begin designing tests, it’s important to have a clear understanding of \nwhy you are testing. This understanding influen

In [52]:
human_message = HumanMessage(content=context + question)
human_message

HumanMessage(content='you respond to the inputs ok. Now how can you define an embedded system based on whatever we\n\nChapter 9: Testing \nEmbedded systems software testing shares much in common with application \nsoftware testing. Thus, much of this chapter is a summary of basic testing \nconcepts and terminology. However, some important differences exist between \napplication testing and embedded systems testing. Embedded developers often \nhave access to hardware-based test tools that are generally not used in application \ndevelopment. Also, embedded systems often have unique characteristics that \nshould be reflected in the test plan. These differences tend to give embedded \nsystems testing its own distinctive flavor. This chapter covers the basics of testing \nand test case development and points out details unique to embedded systems \nwork along the way. \nWhy Test? \nBefore you begin designing tests, it’s important to have a clear understanding of \nwhy you are testing. This 

In [58]:
result = llm.invoke([system_message, human_message])
result

AIMessage(content='Based on the provided context, an embedded system can be defined as a software system that runs reliably for long periods of time, often in applications where human lives are at stake, and is cost-sensitive with little margin for inefficiencies. It must also compensate for problems with the embedded hardware and deal with real-world events that are asynchronous and nondeterministic.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 575, 'total_tokens': 646, 'completion_time': 0.090844478, 'completion_tokens_details': None, 'prompt_time': 0.0366457, 'prompt_tokens_details': None, 'queue_time': 0.157669448, 'total_time': 0.127490178}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019ea174-4927-7421-9617-7cba9ed1a9c6-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input

In [59]:
result.content

'Based on the provided context, an embedded system can be defined as a software system that runs reliably for long periods of time, often in applications where human lives are at stake, and is cost-sensitive with little margin for inefficiencies. It must also compensate for problems with the embedded hardware and deal with real-world events that are asynchronous and nondeterministic.'

In [60]:
question2 = "What is meant by IC?"

In [63]:
docs2 = vectordb.similarity_search_with_score(question2, k=5)
_docs2 = pd.DataFrame(
    [(question, doc[0].page_content, doc[0].metadata.get('source'), doc[0].metadata.get('page'), doc[1]) for doc in
     docs2],
    columns=['query', 'paragraph', 'document', 'page_number', 'relevant_score']
)
context2 = "\n\n".join(_docs2['paragraph'])
human_message = HumanMessage(content=context2 + question2)
result2 = llm.invoke([system_message, human_message])
result2.content

"IC stands for Integrated Circuit, which is a small chip of semiconductor material that contains electronic circuits and components. It's a compact and self-contained unit that can perform various functions, such as processing, memory, and input/output operations. In the context of the provided text, IC technology refers to the advancement in chip design and manufacturing that enables complex systems to be integrated onto a single chip."